# Train the rooftop panel detector

The weights from the earlier project are unrecoverable: the Drive link is dead
and GitHub returns *403, repository exceeded its LFS budget*. The training data
survived, so the model is retrained here instead.

**Why this notebook rather than the original.** The original builds `detectron2`
from source, which was fine in August 2025 and breaks easily against current
Colab torch builds. This uses `torchvision`'s own Mask R-CNN, which is already
installed on Colab and reads the same COCO annotations. Nothing to compile.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. On CPU this
takes days rather than an hour.

**The dataset:** `panel_training_data.zip`, holding `train/`, `valid/` and
`test/`. Put it at the top level of your Google Drive — or, if Drive will not
connect, upload it through the folder icon in the left sidebar. Cell 2 accepts
either and says what to do if it finds neither.

**If Drive fails with "credential propagation was unsuccessful":** that is the
browser blocking third-party cookies, which Colab's Drive login needs. Click
the crossed-out eye icon in the address bar and allow them for this site, then
reload. Or ignore Drive and use the sidebar upload.

In [ ]:
# 1. Check we actually have a GPU. If this says False, fix the runtime first.
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("\nSTOP: Runtime -> Change runtime type -> T4 GPU, then run this again.")

In [ ]:
# 2. Find the dataset and unpack it.
#
# Google Drive is tried first, but Colab's Drive login needs third-party
# cookies and browsers now block those by default - the symptom is
# "credential propagation was unsuccessful". If Drive will not connect this
# falls back to a copy uploaded straight into the session, and says how to
# put one there. Nothing below this cell cares which route was used.
import os, zipfile, pathlib

NAME = 'panel_training_data.zip'
ZIP = None

try:
    from google.colab import drive
    drive.mount('/content/drive')
    for candidate in ('/content/drive/MyDrive/' + NAME,
                      '/content/drive/My Drive/' + NAME):
        if os.path.exists(candidate):
            ZIP = candidate
            break
    if ZIP is None:
        print('Drive connected, but ' + NAME + ' is not at the top level of My Drive.')
except Exception as exc:
    print('Drive would not connect:', exc)
    print('That is almost always third-party cookies being blocked.')
    print('Not a problem - see below.')

if ZIP is None and os.path.exists('/content/' + NAME):
    ZIP = '/content/' + NAME

if ZIP is None:
    print()
    print('Could not find ' + NAME + '. Two ways forward:')
    print()
    print('EITHER fix Drive:')
    print('  click the crossed-out eye icon in the address bar, allow')
    print('  third-party cookies for this site, reload, run this cell again.')
    print()
    print('OR skip Drive entirely:')
    print('  1. click the folder icon in the left sidebar')
    print('  2. click the upload button at the top of that panel')
    print('  3. choose ' + NAME + ' from your laptop')
    print('  4. wait for it to finish, then run this cell again')
    print()
    print('The upload is lost if the session restarts, so Drive is nicer,')
    print('but either one gets you training.')
    raise SystemExit(1)

print('using ' + ZIP)
if not os.path.exists('/content/train'):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content/')

for split in ('train', 'valid', 'test'):
    folder = pathlib.Path('/content') / split
    ann = folder / '_annotations.coco.json'
    print(f"{split:6} {len(list(folder.glob('*.png'))):4} images   annotations: {ann.exists()}")

In [ ]:
# 3. The dataset. COCO polygons are rasterised to masks in the form
# torchvision expects: boxes as xyxy, one mask per instance.
#
# 255 of the 771 training images contain no panels. They are kept, because a
# detector that has never seen an empty roof will invent panels on one, but
# they need empty tensors of the right shape rather than being skipped.
import json
import numpy as np
import torch
from PIL import Image, ImageDraw
from torch.utils.data import Dataset

class PanelDataset(Dataset):
    def __init__(self, folder, train=False):
        self.folder = folder
        self.train = train
        doc = json.load(open(f'{folder}/_annotations.coco.json'))
        self.images = doc['images']
        self.by_image = {}
        for a in doc['annotations']:
            self.by_image.setdefault(a['image_id'], []).append(a)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        info = self.images[i]
        img = Image.open(f"{self.folder}/{info['file_name']}").convert('RGB')
        W, H = info['width'], info['height']
        anns = self.by_image.get(info['id'], [])

        boxes, masks = [], []
        for a in anns:
            x, y, w, h = a['bbox']
            if w < 1 or h < 1:
                continue                      # a box of no width trains nothing
            m = Image.new('L', (W, H), 0)
            draw = ImageDraw.Draw(m)
            for seg in a['segmentation']:
                pts = [(seg[j], seg[j + 1]) for j in range(0, len(seg), 2)]
                if len(pts) >= 3:
                    draw.polygon(pts, fill=1)
            arr = np.array(m, dtype=np.uint8)
            if arr.sum() == 0:
                continue                      # polygon too thin to rasterise
            boxes.append([x, y, x + w, y + h])
            masks.append(arr)

        image = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0

        if self.train and torch.rand(1).item() < 0.5:      # horizontal flip
            image = torch.flip(image, dims=[2])
            masks = [np.fliplr(m).copy() for m in masks]
            boxes = [[W - b[2], b[1], W - b[0], b[3]] for b in boxes]

        target = {
            'boxes': torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            'labels': torch.ones((len(boxes),), dtype=torch.int64),
            'masks': (torch.as_tensor(np.array(masks), dtype=torch.uint8)
                      if masks else torch.zeros((0, H, W), dtype=torch.uint8)),
            'image_id': torch.tensor([info['id']]),
        }
        return image, target

def collate(batch):
    return tuple(zip(*batch))

train_set = PanelDataset('/content/train', train=True)
valid_set = PanelDataset('/content/valid')
test_set  = PanelDataset('/content/test')
print(f"train {len(train_set)}   valid {len(valid_set)}   test {len(test_set)}")

img, tgt = train_set[0]
print("image", tuple(img.shape), img.dtype, "| boxes", tuple(tgt['boxes'].shape),
      "| masks", tuple(tgt['masks'].shape))

In [ ]:
# 4. Sanity check before spending an hour of GPU: draw the labels on an image.
# If the red outlines do not sit on panels, the data is wrong and training
# would only learn the mistake.
import matplotlib.pyplot as plt
import matplotlib.patches as patches

busiest = max(range(len(train_set)),
              key=lambda i: len(train_set.by_image.get(train_set.images[i]['id'], [])))
img, tgt = train_set[busiest]

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img.permute(1, 2, 0))
for b in tgt['boxes']:
    x0, y0, x1, y1 = b.tolist()
    ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                   fill=False, edgecolor='red', linewidth=1.5))
ax.set_title(f"{len(tgt['boxes'])} labelled panel blocks")
ax.axis('off')
plt.show()

In [ ]:
# 5. The model: Mask R-CNN pretrained on COCO, with its two heads replaced for
# our single class. Pretrained weights matter - 771 images is far too few to
# learn edges and textures from nothing.
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

NUM_CLASSES = 2          # background + solarpv

def build_model():
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, NUM_CLASSES)
    in_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_mask, 256, NUM_CLASSES)
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model().to(device)
print("ready on", device)

In [ ]:
# 6. How the model is scored.
#
# This is the same measure the earlier project used, so the numbers stay
# comparable: every predicted panel on an image is merged into one mask, the
# same is done with the labels, and the two are compared as intersection over
# union. It answers "did we find the right area of panel", which is what the
# capacity estimate needs - not "did we count the right number of panels".
#
# An image with no panels and no predictions scores 1.0: being right about an
# empty roof is still being right.
@torch.no_grad()
def mean_iou(model, dataset, score_threshold=0.5, limit=None):
    model.eval()
    scores = []
    n = len(dataset) if limit is None else min(limit, len(dataset))
    for i in range(n):
        image, target = dataset[i]
        out = model([image.to(device)])[0]
        keep = out['scores'] >= score_threshold
        pred = out['masks'][keep]
        pred = (pred.squeeze(1) > 0.5).any(dim=0).cpu().numpy() if len(pred) \
            else np.zeros(image.shape[1:], dtype=bool)
        truth = (target['masks'].numpy().any(axis=0) if len(target['masks'])
                 else np.zeros(image.shape[1:], dtype=bool))
        union = np.logical_or(pred, truth).sum()
        scores.append(1.0 if union == 0 else np.logical_and(pred, truth).sum() / union)
    return float(np.mean(scores))

In [ ]:
# 7. Train. About 45-60 minutes on a T4.
#
# The best epoch by validation IoU is kept, not the last one: the last epoch is
# often slightly overfitted and there is no reason to ship it.
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH = 2                      # 640x640 at batch 4 overflows a T4

train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True,
                          num_workers=2, collate_fn=collate)

params = [p for p in model.parameters() if p.requires_grad]
optimiser = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
schedule = torch.optim.lr_scheduler.StepLR(optimiser, step_size=3, gamma=0.1)

best_iou, history = -1.0, []
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for step, (images, targets) in enumerate(train_loader):
        images = [i.to(device) for i in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        losses = model(images, targets)
        loss = sum(losses.values())

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        running += loss.item()
        if step % 50 == 0:
            print(f"  epoch {epoch+1} step {step:4}/{len(train_loader)} "
                  f"loss {loss.item():.3f}")
    schedule.step()

    iou = mean_iou(model, valid_set)
    history.append((epoch + 1, running / len(train_loader), iou))
    print(f"epoch {epoch+1}: mean loss {running/len(train_loader):.3f}   "
          f"validation IoU {iou:.3f}")

    if iou > best_iou:
        best_iou = iou
        torch.save(model.state_dict(), '/content/panel_detector.pt')
        print(f"   best so far, saved")
print(f"\nbest validation IoU {best_iou:.3f}")

# The run is written to disk the moment it finishes, and pulled to your laptop
# straight away. A Colab runtime can be recycled at any time and takes every
# variable with it; an hour of GPU should not depend on the tab staying open.
import json as _json
with open('/content/train_state.json', 'w') as f:
    _json.dump({'epochs': EPOCHS, 'best_valid_iou': best_iou,
                'train_images': len(train_set), 'valid_images': len(valid_set),
                'test_images': len(test_set),
                'history': [{'epoch': e, 'loss': l, 'valid_iou': v}
                            for e, l, v in history]}, f, indent=2)

try:
    from google.colab import files
    files.download('/content/panel_detector.pt')
    print("panel_detector.pt sent to your Downloads folder.")
except Exception as exc:
    print(f"Could not auto-download ({exc}). Right-click the file in the "
          f"Files panel on the left and choose Download.")

In [ ]:
# 8. Score the kept model on the test set, which it has never seen.
model.load_state_dict(torch.load('/content/panel_detector.pt'))
test_iou = mean_iou(model, test_set)

print(f"TEST IoU {test_iou:.3f}  on {len(test_set)} held-out images")
print()
if test_iou >= 0.75:
    print("Meets the FR-2b threshold of 0.75.")
else:
    print(f"Below the FR-2b threshold of 0.75, by {0.75 - test_iou:.3f}.")
    print("That is a result, not a failure - report where it struggles and why.")
print()
print("Note for the report: this threshold was written for segmenting USABLE")
print("ROOF AREA. This model segments INSTALLED PANELS, which is a different")
print("task. Quote it as what it is.")

In [ ]:
# 9. Look at the predictions. A number alone will not tell you whether the
# model is sensible, and the worst image usually explains more than the mean.
@torch.no_grad()
def show(dataset, index, score_threshold=0.5):
    model.eval()
    image, target = dataset[index]
    out = model([image.to(device)])[0]
    keep = out['scores'] >= score_threshold

    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
    for ax, title in zip(axes, ('labelled by hand', 'predicted')):
        ax.imshow(image.permute(1, 2, 0)); ax.set_title(title); ax.axis('off')
    for b in target['boxes']:
        x0, y0, x1, y1 = b.tolist()
        axes[0].add_patch(patches.Rectangle((x0, y0), x1-x0, y1-y0, fill=False,
                                            edgecolor='red', linewidth=1.5))
    for b, s in zip(out['boxes'][keep].cpu(), out['scores'][keep].cpu()):
        x0, y0, x1, y1 = b.tolist()
        axes[1].add_patch(patches.Rectangle((x0, y0), x1-x0, y1-y0, fill=False,
                                            edgecolor='lime', linewidth=1.5))
        axes[1].text(x0, y0-3, f"{s:.2f}", color='lime', fontsize=8)
    plt.tight_layout(); plt.show()

for i in range(3):
    show(test_set, i)

In [ ]:
# 10. Save the model, and a card recording what it is.
#
# Reads the run from /content/train_state.json rather than from memory, so it
# still works after a runtime restart: the card is provenance, and provenance
# that evaporates when a tab closes is no provenance at all.
import datetime, json as _json

with open('/content/train_state.json') as f:
    state = _json.load(f)

try:
    test_iou_value = test_iou
except NameError:
    raise SystemExit(
        "test_iou is not in memory. Re-run cell 8 first - it reloads the saved "
        "weights and re-scores the test set, which takes under a minute."
    )

card = {
    'trained': datetime.datetime.now().isoformat(timespec='seconds'),
    'architecture': 'torchvision maskrcnn_resnet50_fpn, COCO-pretrained',
    'classes': ['background', 'solarpv'],
    'train_images': state['train_images'],
    'valid_images': state['valid_images'],
    'test_images': state['test_images'],
    'epochs': state['epochs'],
    'best_valid_iou': round(state['best_valid_iou'], 4),
    'test_iou': round(test_iou_value, 4),
    'iou_definition': 'all instances merged per image, then averaged over images',
    'detects': 'installed solar panel blocks, including the gaps between rows',
    'note': 'Replaces weights lost when the original Drive link died and GitHub '
            'LFS returned 403. Retrained from the surviving annotated tiles.',
    'history': [{'epoch': h['epoch'], 'loss': round(h['loss'], 4),
                 'valid_iou': round(h['valid_iou'], 4)}
                for h in state['history']],
}
with open('/content/panel_detector_card.json', 'w') as f:
    _json.dump(card, f, indent=2)

print(_json.dumps({k: v for k, v in card.items() if k != 'history'}, indent=2))

from google.colab import files
files.download('/content/panel_detector.pt')
files.download('/content/panel_detector_card.json')
print()
print("Put panel_detector.pt in the project's data/ folder.")

## When this finishes

Download **`panel_detector.pt`** and **`panel_detector_card.json`** from your
Drive, and put the `.pt` file in the project at `data/panel_detector.pt`.

The card is the provenance: it records what the model is, what it scored and
what it detects. The register rule applies to models too — a number nobody can
trace is a number that gets argued about.

Then the campus census can run: 80 tiles, 0.26 m per pixel, already planned in
`data/campus_tiles_manifest.csv`.